# 02 · Checkpoint 2 experiments

Run this notebook **after** `01_baselines.ipynb` has been completed. It does not change `01_baselines`, its configs, or its outputs.

What it does, in order:

0. **Setup**: mount Drive, get the repo (the branch that contains `scripts/run_experiments.py`), install requirements.
1. **Smoke test**: three tiny runs that check the runner works (F1 will be about 0, that is expected).
2. **Tune**: learning-rate search on seed 1, scored on **English dev only**.
3. **Final runs**: seeds 1 to 3 for `full_ft`, `head_only`, and `lora` at the chosen learning rates.
4. **Table**: mean ± std over seeds, with the cost columns.
5. **Figure**, 6. **Error analysis**, 7. **Paired bootstrap**, 8. **Export** of the evidence files.

Every training phase skips runs whose result file already exists, so after a Colab disconnect you re-run **Setup (section 0)** and then the same phase cell again.

Results are written to `MyDrive/FrozenWeights/results/checkpoint1/`, the same folder `01_baselines` uses. Files from this notebook are named `tune_*` (tuning runs) or `<method>_seed<N>.json` (final runs). A baseline file that used a different learning rate than the chosen one is renamed `stale_*`, not deleted.

## 0. Setup (run first in every new session)

In [ ]:
import os, subprocess, sys
from pathlib import Path

BRANCH = "brindha-test"   # branch containing scripts/run_experiments.py; change to "main" once it is merged
REPO_URL = "https://github.com/Ande404/FrozenWeights.git"
REPO = Path("/content/FrozenWeights")
RESULTS = Path("/content/drive/MyDrive/FrozenWeights/results/checkpoint1")

from google.colab import drive
drive.mount("/content/drive")

def sh(*cmd, check=True):
    subprocess.run(list(cmd), check=check)

if not REPO.exists():
    sh("git", "clone", "-b", BRANCH, REPO_URL, str(REPO))
else:
    sh("git", "-C", str(REPO), "fetch", "origin")
    sh("git", "-C", str(REPO), "checkout", BRANCH)
    sh("git", "-C", str(REPO), "pull", "origin", BRANCH)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"))
# Colab ships an old torchao that makes peft refuse to build LoRA layers. This project does not use it.
sh(sys.executable, "-m", "pip", "uninstall", "-y", "torchao", check=False)

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
RESULTS.mkdir(parents=True, exist_ok=True)
assert (REPO / "scripts" / "run_experiments.py").exists(), f"run_experiments.py not found on branch {BRANCH}"

import torch
print("branch:", BRANCH, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (Runtime > Change runtime type > T4 GPU)")
print("result files so far:", sorted(p.name for p in RESULTS.glob("*_seed?.json")))
if not (RESULTS / "full_ft_seed1.json").exists():
    print("NOTE: full_ft_seed1.json not found. Run notebooks/01_baselines.ipynb (full_ft, seed 1) first.")

## 1. Smoke test (about 5 minutes)

Trains `full_ft`, `head_only`, and `lora` for 3 epochs on 500 English sentences. **F1 around 0.00 is expected**; this only checks that every method runs end to end. If a run fails, the traceback goes to `failures.log` in the results folder.

In [ ]:
!cd /content/FrozenWeights && python scripts/run_experiments.py --phase smoke

In [ ]:
# only if a run failed: show the end of the error log
log = RESULTS / "failures.log"
print(log.read_text()[-3000:] if log.exists() else "no failures.log")

## 2. Tune learning rates (about 30 minutes)

Seed 1, English dev F1 only. Grids: `full_ft` [1e-5, 2e-5, 5e-5], `head_only` [5e-4, 1e-3, 5e-3, 1e-2], `lora` [1e-4, 5e-4, 1e-3]. A baseline file `full_ft_seed1.json` counts as the 2e-5 point. For each method the learning rate with the best English dev F1 is chosen, except that the config default is kept when it is within 0.2 F1 points of the best. The choice is saved to `chosen_lrs.json`.

Do not use the Swahili scores that appear in the tuning output to choose a learning rate.

In [ ]:
!cd /content/FrozenWeights && python scripts/run_experiments.py --phase tune

## 3. Final runs (about 30 minutes)

Seeds 1, 2, 3 for each method at the learning rate in `chosen_lrs.json`. To override a value, add for example `--lr lora=1e-3 head_only=5e-3` to the command.

In [ ]:
!cd /content/FrozenWeights && python scripts/run_experiments.py --phase final

## 4. Results table

Mean ± std over seeds. Swahili columns are zero-shot test scores; cost columns are means over seeds.

In [ ]:
!cd /content/FrozenWeights && python scripts/run_experiments.py --phase table
import pandas as pd
pd.read_csv(RESULTS / "summary_zero_shot.csv", index_col=0).T

## 5. Figure: Swahili F1 vs. trainable parameters

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

names = {"head_only": "Head only", "lora": "LoRA", "full_ft": "Full fine-tuning"}
fig, ax = plt.subplots(figsize=(6, 4))
for m, lab in names.items():
    recs = [json.load(open(p)) for p in sorted(RESULTS.glob(f"{m}_seed?.json"))]
    f1 = [100 * r["metrics"]["sw_test"]["f1"] for r in recs]
    x = recs[0]["cost"]["trainable_params"]
    ax.errorbar(x, np.mean(f1), yerr=np.std(f1, ddof=1) if len(f1) > 1 else 0, fmt="o", capsize=4, label=f"{lab} (n={len(f1)})")
    ax.annotate(f"{np.mean(f1):.1f}", (x, np.mean(f1)), textcoords="offset points", xytext=(6, -12))
ax.set_xscale("log")
ax.set_xlabel("Trainable parameters (log scale)")
ax.set_ylabel("Swahili test F1 (zero-shot)")
ax.set_title("English to Swahili NER, mean ± std over seeds")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS / "fig_f1_vs_params.png", dpi=200)
plt.show()

## 6. Error analysis

Uses the saved seed-1 predictions. Each gold or predicted entity that is wrong is put in one category:
- **missed**: a gold entity with no overlapping prediction
- **spurious**: a predicted entity with no overlapping gold entity
- **wrong type**: right span, wrong type
- **boundary**: overlapping span but different boundaries

In [ ]:
from collections import Counter
from seqeval.metrics.sequence_labeling import get_entities

def load_preds(run):
    return [json.loads(l) for l in open(RESULTS / f"{run}_sw_test_predictions.jsonl", encoding="utf-8")]

def categorize(gold, pred):
    g, p = set(get_entities(gold)), set(get_entities(pred))     # (type, start, end), end inclusive
    errs = []
    for ge in sorted(g - p):
        overlap = [pe for pe in p - g if pe[1] <= ge[2] and ge[1] <= pe[2]]
        if not overlap:
            errs.append(("missed", ge, None))
        else:
            same = [pe for pe in overlap if pe[1:] == ge[1:]]
            errs.append(("wrong type", ge, same[0]) if same else ("boundary", ge, overlap[0]))
    for pe in sorted(p - g):
        if not any(pe[1] <= ge[2] and ge[1] <= pe[2] for ge in g):
            errs.append(("spurious", None, pe))
    return errs

def summarize(run):
    rows, counts, examples = load_preds(run), Counter(), {}
    for i, r in enumerate(rows):
        for cat, ge, pe in categorize(r["gold"], r["pred"]):
            counts[(cat, (ge or pe)[0])] += 1
            examples.setdefault(cat, []).append((i, ge, pe))
    return rows, counts, examples

def show(rows, items, k=5):
    for i, ge, pe in items[:k]:
        toks = rows[i]["tokens"]
        span = lambda e: f"{e[0]} '{' '.join(toks[e[1]:e[2] + 1])}'" if e else "-"
        print(" ".join(toks)[:160])
        print(f"   gold: {span(ge)}   |   predicted: {span(pe)}\n")

In [ ]:
for run in ("full_ft_seed1", "lora_seed1"):
    rows, counts, ex = summarize(run)
    print(f"=== {run}")
    for cat in ("missed", "spurious", "wrong type", "boundary"):
        print(f"{cat:11s}", {t: counts[(cat, t)] for t in ("PER", "ORG", "LOC")})

rows, counts, ex = summarize("full_ft_seed1")
for cat in ("spurious", "wrong type", "boundary", "missed"):
    print(f"##### {cat}")
    show(rows, ex.get(cat, []), k=4)

## 7. Paired bootstrap: is LoRA different from full fine-tuning?

Compares the seed-1 predictions of the two methods on the same resampled test sentences. `delta` is full fine-tuning minus LoRA in F1 points. If the 95% interval contains 0, there is no significant difference in this run. This takes a few minutes; lower `n_resamples` for a quick look.

In [ ]:
from frozenweights import metrics

a, b = load_preds("full_ft_seed1"), load_preds("lora_seed1")
res = metrics.paired_bootstrap([r["gold"] for r in a], [r["pred"] for r in a], [r["pred"] for r in b], n_resamples=500)
lo, hi = res["ci95"]
print(f"full_ft minus lora: {100 * res['delta_f1']:+.2f} F1 points, 95% CI [{100 * lo:+.2f}, {100 * hi:+.2f}], one-sided p = {res['p_value']:.3f}")

## 8. Export the evidence files

Copies the small result files to one Drive folder so you can download them and add them to the repo or the memo.

In [ ]:
import shutil
OUT = Path("/content/drive/MyDrive/FrozenWeights/checkpoint2_evidence")
OUT.mkdir(exist_ok=True)
patterns = ["*_seed?.json", "tune_*_seed1.json", "chosen_lrs.json", "summary_zero_shot.csv", "fig_f1_vs_params.png",
            "full_ft_seed1_sw_test_predictions.jsonl", "lora_seed1_sw_test_predictions.jsonl", "head_only_seed1_sw_test_predictions.jsonl"]
for pat in patterns:
    for p in RESULTS.glob(pat):
        shutil.copy(p, OUT / p.name)
print(len(list(OUT.iterdir())), "files in", OUT)

## Troubleshooting

- **Colab disconnected**: run section 0 again, then the same phase cell. Finished runs are skipped.
- **A run failed**: read the end of `failures.log` (cell in section 1). Send the traceback with the command you ran.
- **`run_experiments.py` not found**: the script is on branch `brindha-test`, not `main`. Check `BRANCH` in section 0.
- **No GPU**: Runtime > Change runtime type > T4 GPU, then re-run section 0.
- **Learning-rate edge**: if a chosen value is the highest in its grid, note it as a limitation; do not keep extending the grid.